# 01 — Core Stop Event Table Pipeline

This notebook runs the full preprocessing pipeline and produces the core stop event table.

**Pipeline steps:**
1. Load raw DVB vehicle position data and timetable
2. Detect stop events (`detector.py`)
3. Expand timetable and compute delay, dwell_time, travel_time (`timetable.py`)
4. Add binary features: `is_peak_hour`, `is_workday`, `has_traffic_signal` (`feature_builder.py`)
5. Save core stop event table

## 0. Imports and paths

In [19]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [20]:
import sys
import time, os, tracemalloc
import polars as pl

# Add project root to path so pipeline modules can be imported
sys.path.append("..")

from pipeline.detector import detect_stop_events
from pipeline.rescue import rescue_dwell_times
from pipeline.timetable import expand_timetable, match_and_compute_delay
from pipeline.feature_builder import add_binary_features

In [21]:
# ── Data paths ────────────────────────────────────────────────
RAW_VEHICLE_PATH   = "../data/regular_lines_0728_0803.parquet"
RAW_TIMETABLE_PATH = "../data/timetable_trips_2025_07_22.csv"
OUTPUT_PATH        = "../data/processed/core_stop_events.parquet"

In [22]:
# ── Performance tracking start ─────────────────────────────────
tracemalloc.start()
pipeline_start = time.time()
_step_times = {}

In [23]:
_t = time.time()
print("Loading raw vehicle position data...")
raw_df = (
    pl.read_parquet(RAW_VEHICLE_PATH)
    .with_columns(
        pl.col("tst_iso").str.to_datetime(format="%Y-%m-%dT%H:%M:%S%.f%z")
    )
    .sort(["fzg_id", "tst_iso"])
    .with_columns(
        pl.int_range(pl.len()).over("fzg_id").alias("row_idx")
    )
)
print(f"Rows:       {len(raw_df):,}")
print(f"Vehicles:   {raw_df['fzg_id'].n_unique():,}")
print(f"Date range: {raw_df['tst_iso'].min()} → {raw_df['tst_iso'].max()}")
_step_times["1. load_vehicle"] = time.time() - _t
print(f"  [{_step_times['1. load_vehicle']:.1f}s]")
raw_df.head(3)

Loading raw vehicle position data...
Rows:       10,708,589
Vehicles:   534
Date range: 2025-07-27 22:00:00.930610+00:00 → 2025-08-03 21:59:59.593713+00:00
  [4.1s]


topic,tst,tst_iso,qos,retain,payloadlen,besetztgrad,distanz,fahrt_id,fzg_id,kurs,lage,linie,linie_text,ort_nr_start,ort_nr_start_vvo,ort_nr_ziel,ort_nr_ziel_vvo,pos_lat,pos_lon,status,status_text,tuerkriterium,zeitstempel,zieltext,row_idx
str,str,"datetime[μs, UTC]",i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,i64,str,i64,str,f64,f64,i64,str,bool,i64,str,i64
"""dvb/itcs/vehicleData/6/positio…","""2025-07-30T00:22:42.626209+020…",2025-07-29 22:22:42.626209 UTC,0,0,349,-1,0,-1,6,1,0,59,null,-1,"""-1""",-1,"""-1""",51.044835,13.653444,1,"""ohneFahrt""",false,0,null,0
"""dvb/itcs/vehicleData/6/positio…","""2025-07-30T00:22:45.626898+020…",2025-07-29 22:22:45.626898 UTC,0,0,349,-1,0,-1,6,1,0,59,null,-1,"""-1""",-1,"""-1""",51.044835,13.653444,1,"""ohneFahrt""",false,0,null,1
"""dvb/itcs/vehicleData/6/positio…","""2025-07-30T00:22:46.625247+020…",2025-07-29 22:22:46.625247 UTC,0,0,349,-1,0,-1,6,1,0,59,null,-1,"""-1""",-1,"""-1""",51.044835,13.653444,1,"""ohneFahrt""",false,0,null,2


In [24]:
_t = time.time()
print("Loading raw timetable...")
raw_tt = pl.read_csv(
    RAW_TIMETABLE_PATH,
    infer_schema_length=10000
)
print(f"Rows:          {len(raw_tt):,}")
print(f"Unique trips:  {raw_tt['fahrt_id'].n_unique():,}")
_step_times["2. load_timetable"] = time.time() - _t
print(f"  [{_step_times['2. load_timetable']:.1f}s]")
raw_tt.head(3)

Loading raw timetable...
Rows:          263,865
Unique trips:  97,982
  [0.3s]


topic,tst,tst_iso,qos,retain,payloadlen,fahrt_id,linie,ort_nr_abfahrt,route_nr,segmente,umlauf_id,wendezeit,zp_abfahrt
str,str,str,i64,i64,i64,f64,f64,f64,f64,str,f64,f64,f64
"""dvb/itcs/timetable/trips/13046…","""2025-07-22T13:25:52.616819+020…","""2025-07-22T11:25:52.616819+00:…",0,1,153,1.3046122e7,4.0,151401.0,254.0,"""[]""",465.0,0.0,1.7532e9
"""dvb/itcs/timetable/trips/86181…","""2025-07-22T13:25:52.616860+020…","""2025-07-22T11:25:52.616860+00:…",0,1,1333,8.618122e6,9.0,183803.0,151.0,"""[{""lenkzeit"": 60, ""ort_nr"": 18…",1337.0,null,1.7532e9
"""dvb/itcs/timetable/trips/31154…","""2025-07-22T13:25:52.616990+020…","""2025-07-22T11:25:52.616990+00:…",0,1,2060,3.1154222e7,63.0,645961.0,342.0,"""[{""lenkzeit"": 120, ""ort_nr"": 6…",6303.0,null,1.7532e9


In [25]:
print("Loading raw timetable...")
raw_tt = pl.read_csv(
    RAW_TIMETABLE_PATH,
    infer_schema_length=10000
)

print(f"Rows:          {len(raw_tt):,}")
print(f"Unique trips:  {raw_tt['fahrt_id'].n_unique():,}")
raw_tt.head(3)

Loading raw timetable...
Rows:          263,865
Unique trips:  97,982


topic,tst,tst_iso,qos,retain,payloadlen,fahrt_id,linie,ort_nr_abfahrt,route_nr,segmente,umlauf_id,wendezeit,zp_abfahrt
str,str,str,i64,i64,i64,f64,f64,f64,f64,str,f64,f64,f64
"""dvb/itcs/timetable/trips/13046…","""2025-07-22T13:25:52.616819+020…","""2025-07-22T11:25:52.616819+00:…",0,1,153,1.3046122e7,4.0,151401.0,254.0,"""[]""",465.0,0.0,1.7532e9
"""dvb/itcs/timetable/trips/86181…","""2025-07-22T13:25:52.616860+020…","""2025-07-22T11:25:52.616860+00:…",0,1,1333,8.618122e6,9.0,183803.0,151.0,"""[{""lenkzeit"": 60, ""ort_nr"": 18…",1337.0,null,1.7532e9
"""dvb/itcs/timetable/trips/31154…","""2025-07-22T13:25:52.616990+020…","""2025-07-22T11:25:52.616990+00:…",0,1,2060,3.1154222e7,63.0,645961.0,342.0,"""[{""lenkzeit"": 120, ""ort_nr"": 6…",6303.0,null,1.7532e9


In [26]:
import json
sample = raw_tt.filter(pl.col("segmente") != "[]")["segmente"][0]
print(json.dumps(json.loads(sample), indent=2))

[
  {
    "lenkzeit": 60,
    "ort_nr": 183901
  },
  {
    "lenkzeit": 60,
    "ort_nr": 184001
  },
  {
    "lenkzeit": 120,
    "ort_nr": 184101
  },
  {
    "lenkzeit": 60,
    "ort_nr": 183601
  },
  {
    "lenkzeit": 60,
    "ort_nr": 184301
  },
  {
    "lenkzeit": 60,
    "ort_nr": 184401
  },
  {
    "lenkzeit": 60,
    "ort_nr": 184501
  },
  {
    "lenkzeit": 60,
    "ort_nr": 184601
  },
  {
    "lenkzeit": 60,
    "ort_nr": 184701
  },
  {
    "lenkzeit": 60,
    "ort_nr": 184801
  },
  {
    "lenkzeit": 60,
    "ort_nr": 184901
  },
  {
    "lenkzeit": 120,
    "ort_nr": 185001
  },
  {
    "lenkzeit": 60,
    "ort_nr": 185301
  },
  {
    "lenkzeit": 60,
    "ort_nr": 185401
  },
  {
    "lenkzeit": 120,
    "ort_nr": 185801
  },
  {
    "lenkzeit": 120,
    "ort_nr": 185601
  },
  {
    "lenkzeit": 60,
    "ort_nr": 185701
  },
  {
    "lenkzeit": 60,
    "ort_nr": 192201
  },
  {
    "lenkzeit": 60,
    "ort_nr": 192203
  },
  {
    "lenkzeit": 60,
    "ort_nr": 191402

In [27]:
_t = time.time()
print("Detecting stop events...")
stop_events = detect_stop_events(raw_df)
print(f"Stop events detected: {len(stop_events):,}")
print("\nstop_status distribution:")
print(
    stop_events
    .group_by("stop_status")
    .len()
    .rename({"len": "count"})
    .with_columns((pl.col("count") / len(stop_events) * 100).round(1).alias("%"))
    .sort("count", descending=True)
)
_step_times["3. detect_stops"] = time.time() - _t
print(f"  [{_step_times['3. detect_stops']:.1f}s]")
stop_events.head(3)

Detecting stop events...
Stop events detected: 1,002,942

stop_status distribution:
shape: (3, 3)
┌─────────────┬────────┬──────┐
│ stop_status ┆ count  ┆ %    │
│ ---         ┆ ---    ┆ ---  │
│ str         ┆ u32    ┆ f64  │
╞═════════════╪════════╪══════╡
│ normal      ┆ 807654 ┆ 80.5 │
│ no_door     ┆ 192683 ┆ 19.2 │
│ multi_door  ┆ 2605   ┆ 0.3  │
└─────────────┴────────┴──────┘
  [181.2s]


fzg_id,drop_row_idx,drop_time,linie,window_lo,window_hi,arrival_time,departure_time,door_open_count,door_close_count,door_near_drop,is_true_multi_door,delta_at_drop,min_distanz,max_distanz,stop_status
i64,i64,"datetime[μs, UTC]",i64,i64,i64,"datetime[μs, UTC]","datetime[μs, UTC]",i64,i64,bool,bool,i64,i64,i64,str
151,106,2025-07-29 02:23:54.604010 UTC,4,102,110,2025-07-29 02:23:41.605618 UTC,2025-07-29 02:23:54.604010 UTC,1,1,true,false,-603,15,618,"""normal"""
151,116,2025-07-29 02:25:44.604703 UTC,4,112,120,2025-07-29 02:25:44.604703 UTC,2025-07-29 02:25:44.604703 UTC,0,0,false,false,-586,20,606,"""no_door"""
151,124,2025-07-29 02:26:37.622760 UTC,4,120,128,2025-07-29 02:26:37.622760 UTC,2025-07-29 02:26:37.622760 UTC,0,0,false,false,-262,60,389,"""no_door"""


In [28]:
_t = time.time()
print("Rescuing dwell=0 events...")
stop_events = rescue_dwell_times(stop_events, raw_df)
print("\nstop_status distribution after rescue:")
print(
    stop_events
    .group_by("stop_status")
    .len()
    .rename({"len": "count"})
    .with_columns((pl.col("count") / len(stop_events) * 100).round(1).alias("%"))
    .sort("count", descending=True)
)
_step_times["4. rescue"] = time.time() - _t
print(f"  [{_step_times['4. rescue']:.1f}s]")

Rescuing dwell=0 events...
[rescue] Case 1 (open found, close missed): 38,791
[rescue] Case 2 (no open found):            163,563
[rescue] Events successfully patched: 171,464

stop_status distribution after rescue:
shape: (3, 3)
┌─────────────┬────────┬──────┐
│ stop_status ┆ count  ┆ %    │
│ ---         ┆ ---    ┆ ---  │
│ str         ┆ u32    ┆ f64  │
╞═════════════╪════════╪══════╡
│ normal      ┆ 792452 ┆ 79.0 │
│ no_door     ┆ 207885 ┆ 20.7 │
│ multi_door  ┆ 2605   ┆ 0.3  │
└─────────────┴────────┴──────┘
  [80.2s]


In [29]:
_t = time.time()
print("Expanding timetable...")
timetable = expand_timetable(raw_tt)
print(f"Stop-level rows:       {len(timetable):,}")
print(f"Unique trips:          {timetable['fahrt_id'].n_unique():,}")
print(f"Unique stops (ort_nr): {timetable['ort_nr'].n_unique():,}")
_step_times["5. expand_timetable"] = time.time() - _t
print(f"  [{_step_times['5. expand_timetable']:.1f}s]")
timetable.head(3)

Expanding timetable...
Stop-level rows:       2,381,003
Unique trips:          97,591
Unique stops (ort_nr): 2,006
  [11.3s]


fahrt_id,stop_index,ort_nr,scheduled_arrival_unix,scheduled_arrival_time
i64,i64,i64,f64,datetime[μs]
16492130,0,162001,1.7539e9,2025-07-30 13:01:00
16492130,1,162101,1.7539e9,2025-07-30 13:02:00
16492130,2,162201,1.7539e9,2025-07-30 13:03:00


In [31]:
_t = time.time()
print("Matching stop events to timetable...")
matched = match_and_compute_delay(stop_events, raw_df, timetable)
total     = len(stop_events)
matched_n = len(matched)
print(f"Total stop events:    {total:,}")
print(f"Successfully matched: {matched_n:,}  ({matched_n/total*100:.1f}%)")
print(f"Unmatched:            {total-matched_n:,}  ({(total-matched_n)/total*100:.1f}%)")
_step_times["6. match_timetable"] = time.time() - _t
print(f"  [{_step_times['6. match_timetable']:.1f}s]")
matched.head(3)

Matching stop events to timetable...
Total stop events:    1,002,942
Successfully matched: 976,393  (97.4%)
Unmatched:            26,549  (2.6%)
  [0.7s]


fzg_id,drop_row_idx,arrival_time,departure_time,linie,fahrt_id,ort_nr_start,stop_index,stop_status,scheduled_arrival_time,delay_calculated_sec,delay_recorded_sec,dwell_time,travel_time,besetztgrad
i64,i64,"datetime[μs, UTC]","datetime[μs, UTC]",i64,i64,i64,i64,str,"datetime[μs, UTC]",f64,i64,f64,f64,i64
151,106,2025-07-29 02:23:41.605618 UTC,2025-07-29 02:23:54.604010 UTC,4,5924129,184104,0,"""normal""",2025-07-29 02:22:00 UTC,101.605618,-10,12.998392,null,1
151,116,2025-07-29 02:25:44.604703 UTC,2025-07-29 02:25:44.604703 UTC,4,5924129,182302,1,"""no_door""",2025-07-29 02:24:00 UTC,104.604703,-20,-1.0,110.000693,0
151,124,2025-07-29 02:26:37.622760 UTC,2025-07-29 02:26:37.622760 UTC,4,5924129,182402,2,"""no_door""",2025-07-29 02:26:00 UTC,37.62276,-20,-1.0,53.018057,0


In [32]:
# Sanity check on computed fields
for field in ["delay_calculated_sec", "dwell_time", "travel_time"]:
    print(f"\n{field} stats (seconds):")
    print(matched.select(field).describe())


delay_calculated_sec stats (seconds):
shape: (9, 2)
┌────────────┬──────────────────────┐
│ statistic  ┆ delay_calculated_sec │
│ ---        ┆ ---                  │
│ str        ┆ f64                  │
╞════════════╪══════════════════════╡
│ count      ┆ 976393.0             │
│ null_count ┆ 0.0                  │
│ mean       ┆ 157.244395           │
│ std        ┆ 546.249572           │
│ min        ┆ -126767.228904       │
│ 25%        ┆ 72.492698            │
│ 50%        ┆ 125.338126           │
│ 75%        ┆ 202.78175            │
│ max        ┆ 189769.060653        │
└────────────┴──────────────────────┘

dwell_time stats (seconds):
shape: (9, 2)
┌────────────┬───────────────┐
│ statistic  ┆ dwell_time    │
│ ---        ┆ ---           │
│ str        ┆ f64           │
╞════════════╪═══════════════╡
│ count      ┆ 976393.0      │
│ null_count ┆ 0.0           │
│ mean       ┆ 32.604596     │
│ std        ┆ 766.033654    │
│ min        ┆ -1.0          │
│ 25%        ┆ 0.0      

In [33]:
_t = time.time()
print("Adding binary features...")
core_table = add_binary_features(matched)
print("is_peak_hour distribution:")
print(core_table.group_by("is_peak_hour").len().sort("is_peak_hour"))
print("\nis_workday distribution:")
print(core_table.group_by("is_workday").len().sort("is_workday"))
print("\nhas_traffic_signal: placeholder (all null — awaiting infrastructure data)")
_step_times["7. add_features"] = time.time() - _t
print(f"  [{_step_times['7. add_features']:.1f}s]")

Adding binary features...
is_peak_hour distribution:
shape: (2, 2)
┌──────────────┬────────┐
│ is_peak_hour ┆ len    │
│ ---          ┆ ---    │
│ i8           ┆ u32    │
╞══════════════╪════════╡
│ 0            ┆ 721637 │
│ 1            ┆ 254756 │
└──────────────┴────────┘

is_workday distribution:
shape: (2, 2)
┌────────────┬────────┐
│ is_workday ┆ len    │
│ ---        ┆ ---    │
│ i8         ┆ u32    │
╞════════════╪════════╡
│ 0          ┆ 391583 │
│ 1          ┆ 584810 │
└────────────┴────────┘

has_traffic_signal: placeholder (all null — awaiting infrastructure data)
  [0.0s]


In [34]:
print("Adding binary features...")
core_table = add_binary_features(matched)

print("is_peak_hour distribution:")
print(core_table.group_by("is_peak_hour").len().sort("is_peak_hour"))

print("\nis_workday distribution:")
print(core_table.group_by("is_workday").len().sort("is_workday"))

print("\nhas_traffic_signal: placeholder (all null — awaiting infrastructure data)")

Adding binary features...
is_peak_hour distribution:
shape: (2, 2)
┌──────────────┬────────┐
│ is_peak_hour ┆ len    │
│ ---          ┆ ---    │
│ i8           ┆ u32    │
╞══════════════╪════════╡
│ 0            ┆ 721637 │
│ 1            ┆ 254756 │
└──────────────┴────────┘

is_workday distribution:
shape: (2, 2)
┌────────────┬────────┐
│ is_workday ┆ len    │
│ ---        ┆ ---    │
│ i8         ┆ u32    │
╞════════════╪════════╡
│ 0          ┆ 391583 │
│ 1          ┆ 584810 │
└────────────┴────────┘

has_traffic_signal: placeholder (all null — awaiting infrastructure data)


## 5. Final schema check and save

In [35]:
_t = time.time()
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
print(f"Saving to {OUTPUT_PATH} ...")
core_table.write_parquet(OUTPUT_PATH)
_step_times["8. save"] = time.time() - _t
print(f"Done. {len(core_table):,} rows saved.  [{_step_times['8. save']:.1f}s]")

Saving to ../data/processed/core_stop_events.parquet ...
Done. 976,393 rows saved.  [0.1s]


In [36]:
# ── Performance Summary ────────────────────────────────────────
total_time = time.time() - pipeline_start
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

raw_size = os.path.getsize(RAW_VEHICLE_PATH) / 1e6
out_size = os.path.getsize(OUTPUT_PATH) / 1e6

print("=" * 52)
print("  Pipeline Performance Summary")
print("=" * 52)
print(f"\n{'Step':<28} {'Time':>8}")
print("-" * 38)
for step, t in _step_times.items():
    print(f"  {step:<26} {t:>7.1f}s")
print("-" * 38)
print(f"  {'TOTAL':<26} {total_time:>7.1f}s")
print(f"\nPeak memory (Python heap): {peak / 1e6:.1f} MB")
print(f"\nFile sizes:")
print(f"  Input  ({os.path.basename(RAW_VEHICLE_PATH)}): {raw_size:.1f} MB")
print(f"  Output ({os.path.basename(OUTPUT_PATH)}):   {out_size:.1f} MB")
print(f"  Compression ratio: {raw_size / out_size:.1f}x")

  Pipeline Performance Summary

Step                             Time
--------------------------------------
  1. load_vehicle                4.1s
  2. load_timetable              0.3s
  3. detect_stops              181.2s
  4. rescue                     80.2s
  5. expand_timetable           11.3s
  6. match_timetable             0.7s
  7. add_features                0.0s
  8. save                        0.1s
--------------------------------------
  TOTAL                        627.2s

Peak memory (Python heap): 3549.8 MB

File sizes:
  Input  (regular_lines_0728_0803.parquet): 342.6 MB
  Output (core_stop_events.parquet):   32.8 MB
  Compression ratio: 10.4x


In [37]:
import os
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

print(f"Saving to {OUTPUT_PATH} ...")
core_table.write_parquet(OUTPUT_PATH)
print(f"Done. {len(core_table):,} rows saved.")

Saving to ../data/processed/core_stop_events.parquet ...
Done. 976,393 rows saved.


In [38]:
import platform, subprocess

print(platform.platform())

result = subprocess.run(
    ["system_profiler", "SPHardwareDataType"],
    capture_output=True, text=True
)
print(result.stdout)

macOS-15.6.1-arm64-arm-64bit
Hardware:

    Hardware Overview:

      Model Name: MacBook Air
      Model Identifier: Mac16,12
      Model Number: MW0X3CH/A
      Chip: Apple M4
      Total Number of Cores: 10 (4 performance and 6 efficiency)
      Memory: 16 GB
      System Firmware Version: 11881.140.96
      OS Loader Version: 11881.140.96
      Serial Number (system): MV6D2GT690
      Hardware UUID: C4DE5A04-59A4-53D3-B209-72E87B806D9C
      Provisioning UDID: 00008132-000A615E2ED9801C
      Activation Lock Status: Enabled




python(26431) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
